# 05 — Statistical Anomaly Baselines

## Objective
Establish transparent robust-statistical baselines using median/MAD before machine learning.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Robust score construction

In [1]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
fraud,_=load_data(); train,val,test,_=chronological_split(fraud); b=HistoryFeatureBuilder().fit(train); Xtr,Xv,Xt=map(b.transform,[train,val,test]); Xv["value_robust_z"]=robust_z(Xv.purchase_value); Xv["age_robust_z"]=robust_z(Xv.account_age_hours); Xv["stat_score"]=np.maximum(abs(Xv.value_robust_z),abs(Xv.age_robust_z)); display(Xv.sort_values("stat_score",ascending=False)[["user_id","purchase_value","account_age_hours","stat_score","class"]].head(25)); print(ranking_metrics(Xv['class'],Xv.stat_score))

,user_id,purchase_value,account_age_hours,stat_score,class
112255,30719,129,2124.338333,4.877154,0
36677,126966,128,1378.586667,4.825269,0
95004,10557,127,2511.251389,4.773385,0
84066,367779,122,2310.286944,4.513962,0
27774,133743,118,453.631667,4.306423,0
84986,173542,116,2760.017222,4.202654,0
83869,324166,116,1899.924722,4.202654,0
112512,391333,113,869.500833,4.047000,0
2551,59984,112,2595.286944,3.995115,0
94121,386343,112,2428.657222,3.995115,0


{'pr_auc': 0.045679823037796, 'roc_auc': 0.4951346874099683, 'precision_at_50': 0.0, 'recall_at_50': 0.0, 'precision_at_100': 0.03, 'recall_at_100': 0.0028846153846153848, 'precision_at_500': 0.044, 'recall_at_500': 0.021153846153846155}


## 2. Baseline performance

In [2]:
fig=px.scatter(Xv.sample(min(15000,len(Xv)),
                         random_state=42),x="purchase_value",
                         y="account_age_hours",color="stat_score",
                         hover_data=["user_id","device_id"],
                         title="Statistical anomaly landscape"); fig.show()

## 3. Interactive review cutoff

In [3]:
pct=widgets.IntSlider(value=99,min=90,max=99,description="Percentile"); out=widgets.Output()
def review(*_):
    q=np.percentile(Xv.stat_score,pct.value)
    with out: out.clear_output(); display(Markdown(f"**Cutoff:** {q:.3f} | **Manual-review queue:** {(Xv.stat_score>=q).sum():,}"))
pct.observe(review,'value'); display(pct,out); review()

IntSlider(value=99, description='Percentile', max=99, min=90)

Output()